# L2P + Quantum Interference Prompt Gating — Split CIFAR-100

L2P keeps its top-k prompt selection by cosine similarity. After selection, each prompt is rescaled as $\widetilde P_i = k\,w_i P_i$ with

$$a_i(x)=\sqrt{\operatorname{softmax}(s_i/\tau)}\,e^{\mathrm{i}\phi_i(x)},\qquad
w_i(x)=\frac{|[U_\theta a(x)]_i|^2}{\sum_{j\le k}|[U_\theta a(x)]_j|^2}$$

where $U_\theta$ is a 3-qubit circuit of rotations and CNOTs, simulated exactly.

Variants (same data, seed, loss and evaluation; only `--prompt_gating` changes):

| `--prompt_gating` | what it does |
|---|---|
| `none` | original L2P |
| `softmax` | $w=\operatorname{softmax}(s/\tau)$, no parameters |
| `quantum` | 3-qubit interference gate (phases from the image CLS feature) |
| `linear` | classical gate with the same parameter budget: $w=\operatorname{softmax}(s/\tau + u_j^\top q(x) + b_j)$ |

**Notebook settings:** Accelerator **GPU T4 ×2** (a P100 also works, with runs executed one after another), **Internet on**.
Use *Save Version → Save & Run All* so the runs keep going in the background (12 h limit per session).

Code: the notebook uses the repo attached under `/kaggle/input` if there is one (upload the repo folder as a Dataset); otherwise it clones `REPO_URL`/`BRANCH`.
CIFAR-100: an attached dataset that contains `cifar-100-python/` is used if found; otherwise it is downloaded.

In [ ]:
# ---- configuration ----
REPO_URL = 'https://github.com/thanhphu25/l2p-pytorch.git'
BRANCH = 'quantum-gating'
CONFIG = 'cifar100_l2p'                              # or 'five_datasets_l2p'
VARIANTS = ['none', 'softmax', 'quantum', 'linear']  # scheduled in this order; 'linear' is the param-matched classical control
SEEDS = [42]                                         # add seeds (e.g. [42, 43, 44]) if the time budget allows
AMP = True                                           # fp16 autocast, much faster on T4; False = fp32 exactly like the original repo
EXTRA_ARGS = []                                      # passed to main.py, e.g. ['--epochs', '1'] for a quick check or ['--gate_tau', '0.2']

CODE_DIR = '/tmp/l2p-pytorch'
DATA_PATH = '/tmp/data'
OUT_DIR = '/kaggle/working/results'

In [ ]:
import glob, os, shutil, subprocess, sys

def run(cmd, cwd=None):
    """Run a command and stream its output into the notebook."""
    print('$', ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'command failed with exit code {proc.returncode}')

def find_in_inputs(pattern, max_depth=5):
    for depth in range(1, max_depth + 1):
        hits = sorted(glob.glob(os.path.join('/kaggle/input', *['*'] * depth, pattern)))
        if hits:
            return hits
    return []

# code: attached dataset first, then git
shutil.rmtree(CODE_DIR, ignore_errors=True)
attached = [os.path.dirname(p) for p in find_in_inputs('prompt.py')
            if os.path.exists(os.path.join(os.path.dirname(p), 'main.py'))]
if attached:
    print('using code from', attached[0])
    shutil.copytree(attached[0], CODE_DIR)
else:
    run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, CODE_DIR])
assert 'class QuantumGate' in open(os.path.join(CODE_DIR, 'prompt.py')).read(), \
    'this copy of the repo has no prompt gating: push the branch or attach the updated repo as a dataset'

run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==0.6.7'])
run(['nvidia-smi'])

In [ ]:
# CIFAR-100: reuse an attached copy if there is one, otherwise download; then cache the ViT-B/16 (ImageNet-21k) weights
os.makedirs(DATA_PATH, exist_ok=True)
if CONFIG == 'cifar100_l2p' and not os.path.exists(os.path.join(DATA_PATH, 'cifar-100-python')):
    found = [os.path.dirname(p) for p in find_in_inputs('cifar-100-python/meta')]
    if found:
        print('using CIFAR-100 from', found[0])
        os.symlink(found[0], os.path.join(DATA_PATH, 'cifar-100-python'))
if CONFIG == 'cifar100_l2p':
    run([sys.executable, '-c', f"import torchvision as tv; [tv.datasets.CIFAR100({DATA_PATH!r}, train=t, download=True) for t in (True, False)]"])
else:  # download once here so parallel runs do not race on the same files
    run([sys.executable, '-c', "import argparse, datasets; a = argparse.Namespace(data_path=%r); "
         "[datasets.get_dataset(n, None, None, a) for n in ['SVHN', 'MNIST', 'CIFAR10', 'NotMNIST', 'FashionMNIST']]" % DATA_PATH],
        cwd=CODE_DIR)
run([sys.executable, '-c', "import timm, models; timm.create_model('vit_base_patch16_224', pretrained=True); print('weights cached')"],
    cwd=CODE_DIR)

In [ ]:
# quick checks of the gates (circuit vs an independent numpy reference, init == softmax, a tiny end-to-end run)
run([sys.executable, '-W', 'ignore', 'tests/test_prompt_gating.py'], cwd=CODE_DIR)

## Run the comparison
One run per GPU at a time. Progress (current iteration and ETA of each run) is printed every 5 minutes.
Each run writes `train.log` and `summary.json` to `OUT_DIR/<variant>_s<seed>/`. `summary.json` is rewritten after every task, and completed runs are skipped if this cell is started again.

In [ ]:
cmd = [sys.executable, 'kaggle/run_experiments.py', '--config', CONFIG, '--data-path', DATA_PATH, '--out', OUT_DIR,
       '--variants', *VARIANTS, '--seeds', *map(str, SEEDS)] + (['--amp'] if AMP else []) + EXTRA_ARGS
run(cmd, cwd=CODE_DIR)

## Results
*Final avg acc* is L2P's "Average Acc" (mean accuracy over all tasks after the last task). *Avg incremental acc* averages that quantity over all 10 stages. *Forgetting* uses L2P's definition. *Gate dev.* is the mean L1 distance between the learned gate weights and the plain $\operatorname{softmax}(s/\tau)$ on the test sets after the last task: a value near 0 means the gate learned nothing beyond the cosine prior.

In [ ]:
run([sys.executable, 'kaggle/summarize.py', '--root', OUT_DIR], cwd=CODE_DIR)
from IPython.display import Markdown, display
display(Markdown(open(os.path.join(OUT_DIR, 'results.md')).read()))